In [1]:
from lignin_saf.ligsaf_chemicals import create_chemicals
from lignin_saf.settings.process_params import feed_parameters, solvolysis_params, hydrogenolysis_params, rcf_oil_yield, additional_rcf, etoac_purification, hexane_purification
from lignin_saf.settings.prices import prices,  _feedstock_price_dry_ton, kg_per_ton, h2_price
from lignin_saf.settings.tea_params import operating_days, labor
from lignin_saf.systems.rcf import create_rcf_system
from lignin_saf.systems.cellulosic_ethanol_no_preatreatment import create_cellulosic_ethanol_system
from lignin_saf.cellulosic_tea import create_cellulosic_ethanol_tea
from lignin_saf.systems.rcf_oil_purification import create_rcf_oil_purification_system
from lignin_saf.systems.monomer_purification import create_monomer_purification_system
from lignin_saf.ligsaf_units import HydrogenStorageTank


from biosteam import main_flowsheet as F
import biosteam as bst

chems = create_chemicals()
bst.settings.set_thermo(chems)
bst.settings.CEPCI = 840

chems.define_group(
    name='Poplar',
    IDs=['Glucan', 'Xylan', 'Arabinan', 'Mannan', 'Galactan',
         'Sucrose', 'Lignin', 'Acetate', 'Extract', 'Ash'],
    composition=[0.464, 0.134, 0.002, 0.037, 0.014,
                 0.001, 0.285, 0.035, 0.016, 0.012],
    wt=True
)

poplar_in = bst.Stream('Poplar_In',
                       Poplar=feed_parameters['flow'] * 1e3,
                       Water=feed_parameters['moisture'] * feed_parameters['flow'] * 1e3,
                       phase='l', units='kg/d', price=prices['Feedstock'])

# ── Area 200: RCF process ──────────────────────────────────────────────────
rcf_system = create_rcf_system(ins=poplar_in)
rcf_system.simulate()

rcf_oil_purification_sys = create_rcf_oil_purification_system(ins=F.RCF_CRUDE_OUT)
rcf_oil_purification_sys.simulate()


monomer_purification_sys = create_monomer_purification_system(ins=F.PURE_OIL_OUT)
monomer_purification_sys.simulate()


# ── Cellulosic ethanol — Carbohydrate_Pulp feeds directly into fermentation ─
etoh_system = create_cellulosic_ethanol_system(ins=F.Carbohydrate_Pulp)
etoh_system.simulate()

# No pretreatment_wastewater — only S401 stillage filtrate goes to WWT.
etoh_ww     = [F.unit.S401.outs[1]]
etoh_solids = [F.unit.S401.outs[0]]

# ── WWT: RCF wastewater + ethanol stillage filtrate ────────────────────────
WWT = bst.create_conventional_wastewater_treatment_system(
    'WWT',
    ins=[F.RCF_WW_OUTS, F.WW_10, F.WastePulp, F.WW_11, F.WW_12] + etoh_ww,
)
for unit in WWT.units:
    if hasattr(unit, 'strict_moisture_content'):
        unit.strict_moisture_content = False

# Wire WWT RO-treated water to PWC; create_all_facilities(WWT=False) leaves
# M2 (placeholder mixer) empty, so PWC would otherwise buy ~480,000 kg/hr
# of fresh water unnecessarily.
F.unit.PWC.ins[0] = WWT.outs[2]

solids_to_BT = bst.Mixer('MIX_BT_solids', ins=[WWT.outs[1]] + etoh_solids)
gas_mixer    = bst.Mixer('MIX_BT_gas',    ins=[F.RCF_PSAWASTE_OUTS, WWT.outs[0]])

BT = bst.facilities.BoilerTurbogenerator('BT', fuel_price=prices['CH4'])
BT.ins[0] = solids_to_BT.outs[0]
BT.ins[1] = gas_mixer.outs[0]



h2_rcf = bst.Stream()
h2_rcf.copy_like(F.RCF_H2_IN)

shared_h2_storage = HydrogenStorageTank('H2_TK', ins=h2_rcf)



rcf_monomers_system = bst.System(
    'RCF_OIL_system',
    path=(rcf_system, rcf_oil_purification_sys, monomer_purification_sys, etoh_system, WWT),
    facilities=[solids_to_BT, gas_mixer, BT, shared_h2_storage],
)

rcf_monomers_system.simulate()
integrated_tea = create_cellulosic_ethanol_tea(rcf_monomers_system)
F.ethanol.price = 0.8822293394025529


F.cellulase.price = prices['Cellulase'] 
F.CSL.price = prices ['CSL'] 
F.DAP.price = prices['DAP'] 
F.caustic.price = prices['Caustic']
F.denaturant.price =  prices['Denaturant'] 
F.cooling_tower_chemicals.price = prices['CT_chemicals'] 





c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Methane has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\dew_point.py:129: RuntimeWarning: Methane has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\_pump.py:224: RuntimeWarning: <Pump: RCF_PUMP1> no pump type available at current power (2.46e+03 hp), head (3.36e+03 ft), kinematic viscosity (6.09e-07 m2/s), and NPSH (0.746 ft); assuming centrigugal pump
  warn(f'{repr

In [2]:
# Labor cost from [1]
# [1] W. Seider et al., Product and Process Design Principles. 2016. John Wiley & Sons.
# Table 17.3 from [1]
# For RCF - 6 operators for the reactors (solids-fluids processing, and > 100 ton/day so 3 x 2 =6 ), and 2 for the distillation columns downstream for solvent recovery. Total operators: 8
# For oil purification + monomer purification 2 operators  each Total operator: 4

# For ethanol production, 6 operators for reactors as solids-fluids processing and large volumes so 3 x 2 = 6, and then 2 for the beer column downstream for ethanol purification. Total operators: 8
# 1 operators for the storage. Total operators: 1
# 1 operators for WWT (I think complexity of feed streams), and 1 for BT. Total operator: 2
# Total operators per shift: 23

num_operators_per_shift = 23
num_shifts              = 5       # number of operator shifts (4 working + 1 relief)
pay_rate                = 40      # [USD/hr] operator base pay rate

DWandB             = num_operators_per_shift * num_shifts * 2080 * pay_rate
Dsalaries_benefits = 0.15 * DWandB          # 15% of DW&B for salaried staff + benefits
O_supplies         = 0.06 * DWandB          # 6% of DW&B for operating supplies
technical_assistance = 5 * 75_000           # 5 technical staff @ $75,000/yr
control_lab          = 5 * 80_000           # 5 lab/QC staff  @ $80,000/yr

labor = DWandB + Dsalaries_benefits + O_supplies + technical_assistance + control_lab
# [USD/yr] total annual labor cost; passed to tea.labor_cost after creating the TEA object

In [3]:
integrated_tea.operating_days = 330
integrated_tea.labor_cost = labor

In [4]:

msp = round((integrated_tea.solve_price(F.MON_MONOMERS_OUT)),2)
print(f'The MSP for RCF crude is  {msp} USD/kg')

The MSP for RCF crude is  5.46 USD/kg


In [5]:
chems['Methanol'].Psat(T=225+273.15)

6318477.618571473

In [6]:
F.RCF_RXR1.results()

Solvolysis reactor                                  Units  RCF_RXR1
Electricity         Power                              kW      44.2
                    Cost                           USD/hr      3.46
Design              Diameter                           ft      11.9
                    Length                             ft      59.7
                    Reactor volume                     m3       189
                    Total volume                       m3       756
                    Total beds                                    4
                    Beds in service                               3
                    Time on stream                     hr         3
                    Residence time                     hr       0.3
                    Turnaround time                    hr         1
                    Batch time                         hr         4
                    Biomass volume per bed             m3       172
                    Solvent volume per bed             m3      94.5
                    Instantaneous loading            L/kg      1.13
                    Solvent loading                  L/kg      11.3
                    Vessel type                            Vertical
                    Weight                             lb  6.33e+05
                    Wall thickness                     in      5.97
                    Pressure drop                     bar    0.0132
Purchase cost       Vertical pressure vessel (x4)     USD  1.08e+07
                    Platform and ladders (x4)         USD  2.74e+05
                    Pump 1 - Pump (x4)                USD  2.48e+05
                    Pump 1 - Motor (x4)               USD  3.89e+03
Total purchase cost                                   USD  1.14e+07
Utility cost                                       USD/hr      3.46

In [7]:
model = bst.Model(rcf_monomers_system)

In [8]:
from chaospy import distributions as shape
param = model.parameter

In [9]:

var_50 = 0.5 # 50% variation in parameters - set for a few
var_20 = 0.2 # 20% variation in other parameters

In [10]:
# Operating days — kind='isolated': only affects TEA time scaling, not mass/energy balance
dist = shape.Uniform(lower = 297 , upper = 363)
@param(name = 'Operating days',
    element = 'Overall',
    kind = 'isolated',
    units = 'days',
    baseline = integrated_tea.operating_days,
    distribution = dist)
def set_opertaing_days(i):
    integrated_tea.operating_days = i


# Poplar feedstock
dist = shape.Uniform(lower = 50 , upper = 100)
@param(name = 'Poplar feedstock price',
    element = 'Overall',
    kind = 'isolated',
    units = 'USD/DMT',
    baseline = _feedstock_price_dry_ton,
    distribution = dist)
def set_poplar_feedstock_price(i):
    # i is in USD/dry short ton; stream price must be USD/kg wet biomass
    F.Poplar_In.price = i / kg_per_ton / (1 + feed_parameters['moisture'])



# Labor cost
dist = shape.Uniform(lower = integrated_tea.labor_cost * (1-var_50) , upper = integrated_tea.labor_cost * (1+var_50) )
@param(name = 'Labor cost',
    element = 'Overall',
    kind = 'isolated',
    units = 'USD/yr',
    baseline = integrated_tea.labor_cost,
    distribution = dist)
def set_labor_cost(i): 
    integrated_tea.labor_cost = i


# Electricity price
dist = shape.Triangle(lower = 0.0641, midpoint = bst.settings.electricity_price, upper = 0.1798) 
@param(name = 'Electricity cost',
    element = 'Overall', 
    kind = 'isolated',
    units = 'USD/kWh',
    baseline = bst.settings.electricity_price, distribution = dist)
def set_electricity_price(i):
    bst.settings.electricity_price = i  



# Hydrogen price
dist = shape.Triangle(lower = 2.74 , midpoint=3.70, upper = 11.53)
@param(name = 'Hydrogen price',
       element = 'Overall',
       kind = 'isolated',
       units = 'USD/kg',
       baseline =  h2_price,
       distribution = dist)
def set_hydrogen_price(i):
    F.RCF_H2_IN.price = i


# Hydrogen storage time
dist = shape.Uniform(lower = 0.25 , upper = 3)
@param(name = 'Hydrogen storage period',
       element = 'Overall',
       kind = 'isolated',
       units = 'days',
       baseline =  F.H2_TK.storage_period,
       distribution = dist)
def set_h2_storage_period(i):
    F.H2_TK.storage_period = i
    F.H2_TK.simulate()   # re-runs _design()/_cost() so CAPEX updates for solve_price()


In [11]:
## Solvolysis pressure
#dist = shape.Uniform(lower = solvolysis_params['P'],  upper = 83.3e5)
#@param(name = 'Solvolysis reactor pressure',
#    element = 'RCF', 
#    kind = 'coupled',
#    units = 'Pa',
#    baseline = solvolysis_params['P'], distribution = dist)
#def set_solvolysis_rxr_pressure(i):
#    F.RCF_RXR1.P = i  
#


# Solvolysis reaction time
dist = shape.Triangle(lower = 3*(1-var_50), midpoint = 3, upper = 3*(1+var_50))
@param(name = 'Solvolysis reaction time',
    element = 'RCF', 
    kind = 'coupled',
    units = 'hr',
    baseline = solvolysis_params['tau_s'], distribution = dist)
def set_solvolysis_reaction_time(i):
    F.RCF_RXR1.tau = i      


# Solvolysis residence time
dist = shape.Triangle(lower = 9/60, midpoint = solvolysis_params['tau_s_res'], upper = 36/60) 
@param(name = 'Solvolysis residence time',
    element = 'RCF', 
    kind = 'coupled',
    units = 'hr',
    baseline = solvolysis_params['tau_s_res'], distribution = dist)
def set_solvolysis_residence_time(i):
    F.RCF_RXR1.tau_residence = i  


# Solvolysis cleaning time
dist = shape.Triangle(lower = 0.5, midpoint = 1, upper = 2) 
@param(name = 'Solvolysis cleaning time',
    element = 'RCF', 
    kind = 'coupled',
    units = 'hr',
    baseline = F.RCF_RXR1.tau_0, distribution = dist)
def set_solvolysis_cleaning_time(i):
    F.RCF_RXR1.tau_0 = i  


# Methanol price
dist = shape.Triangle(lower = 0.2648, midpoint = F.RCF_MEOH_IN.price, upper = 0.3972)
@param(name = 'Methanol cost',
    element = 'RCF', 
    kind = 'isolated',
    units = 'USD/kg',
    baseline = F.RCF_MEOH_IN.price, distribution = dist)
def set_methanol_price(i):
    F.RCF_MEOH_IN.price = i  

# Solvent losses in pulp
dist = shape.Uniform(lower=0.005, upper=0.05)
@param(name='RCF solvent losses',
       element='RCF',
       kind='coupled',
       units='-',
       baseline=solvolysis_params['solvent_losses'],
       distribution=dist)
def set_solvent_losses(i):
    solvolysis_params['solvent_losses'] = i

# RCF reactor duty
dist = shape.Uniform(lower=49, upper=72)
@param(name='RCF reactor duty',
       element='RCF',
       kind='coupled',
       units='-',
       baseline=hydrogenolysis_params['duty'],
       distribution=dist)
def set_rcf_reactor_duty(i):
    hydrogenolysis_params['duty'] = i


# RCF catalyst cost 
dist = shape.Uniform(lower = F.RCF_CAT_IN.price * (0.1), upper = F.RCF_CAT_IN.price * (10) )  # Same variation as Bartling et al
@param(name = 'RCF catalyst price',
    element = 'RCF', 
    kind = 'isolated',
    units = 'USD/kg',
    baseline = F.RCF_CAT_IN.price, distribution = dist)
def set_rcf_catalyst_price(i):
    F.RCF_CAT_IN.price = i      



# RCF catalyst lifetime 
dist = shape.Uniform(lower = 1, upper = 36)  # Same as Bartling et al
@param(name = 'RCF catalyst lifetime',
    element = 'RCF', 
    kind = 'isolated',
    units = 'months',
    baseline = solvolysis_params['cat_lifetime'], distribution = dist)
def set_rcf_catalyst_lifetime(i):
    solvolysis_params['cat_lifetime'] = i
    F.RCF_CAT_IN.imass['NiC'] = (
        solvolysis_params['cat_loading'] * (feed_parameters['flow'] * 1e3 / 24) * solvolysis_params['tau_h']
    ) / (i * 30 * 24)   # [kg/hr]


# RCF catalyst loading 
dist = shape.Uniform(lower = solvolysis_params['cat_loading'] * (1-var_50) , upper = solvolysis_params['cat_loading'] * (1+var_50) ) 
@param(name = 'RCF catalyst loading',
    element = 'RCF', 
    kind = 'isolated',
    units = 'kg/kg-biomass',
    baseline = solvolysis_params['cat_loading'], distribution = dist)
def set_rcf_catalyst_loading(i):
    solvolysis_params['cat_loading'] = i 
    F.RCF_CAT_IN.imass['NiC'] = (
        i * (feed_parameters['flow'] * 1e3 / 24) * solvolysis_params['tau_h']
    ) / (solvolysis_params['cat_lifetime'] * 30 * 24)   # [kg/hr]

# Cellulose retention
dist = shape.Triangle(lower=0.8, midpoint = solvolysis_params['Cellulose_retention'], upper=1.0)
@param(name='Cellulose retention',
       element='RCF',
       kind='coupled',
       units='%',
       baseline=solvolysis_params['Cellulose_retention'],
       distribution=dist)
def set_cellulose_retention(i):
    solvolysis_params['Cellulose_retention'] = i


# Xylose retention
dist = shape.Triangle(lower=0.2, midpoint = solvolysis_params['Xylose_retention'], upper=1.0)
@param(name='Xylose retention',
       element='RCF',
       kind='coupled',
       units='%',
       baseline=solvolysis_params['Xylose_retention'],
       distribution=dist)
def set_xylose_retention(i):
    solvolysis_params['Xylose_retention'] = i


# Delignification
dist = shape.Triangle(lower = 0.24 , midpoint = solvolysis_params['Delignification'], upper = 0.61)
@param(name = 'Delignfication',
       element = 'RCF',
       kind = 'coupled',
       units = '%',
       baseline = solvolysis_params['Delignification'], distribution = dist)
def set_delignfication(i):
    solvolysis_params['Delignification'] = i
    F.RCF_RXR1.reaction_1.X = i   


# Condensation extent
dist = shape.Triangle(lower = 0.001 , midpoint = hydrogenolysis_params['condensation_extent'], upper = 0.709)
@param(name = 'Condensation extent',
       element = 'RCF',
       kind = 'coupled',
       units = '%',
       baseline = hydrogenolysis_params['condensation_extent'], distribution = dist)
def set_condensation_extent(i):
    _X_scale = 1.0 - 1e-6
    hydrogenolysis_params['condensation_extent'] = i
    # X values are baked into the ParallelReaction at create_rcf_system() time; must update directly.
    # Reactions 0 & 1: Propylguaiacol/Propylsyringol (monomers that escaped condensation).
    # Reactions 4 & 5: S_Oligomer/G_Oligomer (baseline oligomers + condensed monomer fraction).
    F.RCF_RXR2.reaction[0].X = _X_scale * rcf_oil_yield['Monomers'] * 0.5 * (1 - i)
    F.RCF_RXR2.reaction[1].X = _X_scale * rcf_oil_yield['Monomers'] * 0.5 * (1 - i)
    F.RCF_RXR2.reaction[4].X = _X_scale * (rcf_oil_yield['Oligomers'] * 0.5 + rcf_oil_yield['Monomers'] * 0.5 * i)
    F.RCF_RXR2.reaction[5].X = _X_scale * (rcf_oil_yield['Oligomers'] * 0.5 + rcf_oil_yield['Monomers'] * 0.5 * i)


# Distillation col 1 light key recovery at the top
dist = shape.Uniform(lower = 0.7, 
                     upper = 0.9999) 
@param(name = 'Light key recovery - column 1',
    element = 'RCF', 
    kind = 'coupled',
    units = 'wt%',
    baseline = additional_rcf['rcf_col_1_light_dist_recovery'], distribution = dist)
def set_light_key_recovery_column_1(i):
    additional_rcf['rcf_col_1_light_dist_recovery'] = i
    F.unit.RCF_COL1.Lr = i



In [12]:
# Glucose to ethanol conversion
dist = shape.Triangle(lower = 0.8, midpoint = F.R303.cofermentation[0].X, upper = 0.95)
@param(name = 'Glucose to ethanol conv.',
    element = 'EHF', 
    kind = 'coupled',
    units = '-',
    baseline = F.R303.cofermentation[0].X, distribution = dist)
def set_glucose_to_ethanol_conv(i):
    F.R303.cofermentation[0].X = i  

# Glucan to glucose conversion 
dist = shape.Triangle(lower = 0.8, midpoint = F.R303.saccharification[2].X,  upper = 0.9)
@param(name = 'Glucan to glucose conv.',
    element = 'EHF', 
    kind = 'coupled',
    units = '-',
    baseline = F.R303.saccharification[2].X, distribution = dist)
def set_glucan_to_glucose_conv(i):
    F.R303.saccharification[2].X = i  


# Saccharification residence time
dist = shape.Triangle(lower = 60 * (1-var_20), midpoint = F.R303.tau_saccharification, upper = 60 * (1+var_20))
@param(name = 'Saccharification residence time',
    element = 'EHF', 
    kind = 'coupled',
    units = 'hr',
    baseline = F.R303.tau_saccharification, distribution = dist)
def set_saccharification_tau(i):
    F.R303.tau_saccharification = i  

# Cofermentation residence time
dist = shape.Triangle(lower = 36 * (1-var_20),  midpoint = F.R303.tau_cofermentation, upper = 36 * (1+var_20))
@param(name = 'Cofermentation residence time',
    element = 'EHF', 
    kind = 'coupled',
    units = 'hr',
    baseline = F.R303.tau_cofermentation, distribution = dist)
def set_cofermentation_tau(i):
    F.R303.tau_cofermentation = i  


# Xylose to ethanol conversion
dist = shape.Triangle(lower = 0.8, midpoint =  F.R303.cofermentation[4].X, upper = 0.9)
@param(name = 'Xylose to ethanol conv.',
    element = 'EHF', 
    kind = 'coupled',
    units = '-',
    baseline = F.R303.cofermentation[4].X, distribution = dist)
def set_xylose_to_ethanol_conv(i):
    F.R303.cofermentation[4].X = i      

# Xylan to xylose conversion
dist = shape.Triangle(lower=0.8, midpoint = F.unit.R301.reactions[0].X , upper=0.9)
@param(name='Xylan to xylose conversion',
       element='Cellulosic ethanol',
       kind='coupled',
       units='-',
       baseline=F.unit.R301.reactions[0].X,
       distribution=dist)
def set_xylan_to_xylose_conversion(i):
    F.unit.R301.reactions[0].X = i



# Cellulase enzyme loading
dist = shape.Triangle(lower = 0.01, midpoint = F.M301.enzyme_loading, upper = 0.05)
@param(name = 'Cellulase enzyme loading',
    element = 'EHF', 
    kind = 'coupled',
    units = 'wt%',
    baseline = F.M301.enzyme_loading, distribution = dist)
def set_enzyme_loading(i):
    F.M301.enzyme_loading = i  


# Cellulase price
dist = shape.Uniform(lower = F.M301.ins[1].price * (1-var_50), upper = F.M301.ins[1].price * (1+var_50))
@param(name = 'Cellulase price',
    element = 'EHF',
    kind = 'isolated',
    units = 'USD/kg',
    baseline = F.M301.ins[1].price, distribution = dist)
def set_cellulase_price(i):
    F.M301.ins[1].price = i

# Ethanol co-product revenue 
dist = shape.Uniform(lower = F.ethanol.price*(1-var_50), upper = F.ethanol.price*(1+var_50))
@param(name = 'Ethanol price',
    element = 'EHF',
    kind = 'isolated',
    units = 'USD/kg',
    baseline = F.ethanol.price, distribution = dist)
def set_ethanol_price(i):
    F.ethanol.price = i

    


In [13]:
# Etyl acetate solvent to crude RCF oil ratio
dist = shape.Uniform(lower=etoac_purification['solvent_to_crude_ratio'] * (1 - var_20),
                     upper=etoac_purification['solvent_to_crude_ratio'] * (1 + var_20))
@param(name='EtOAc solvent to crude ratio',
       element='OP',
       kind='coupled',
       units='L/kg',
       baseline=etoac_purification['solvent_to_crude_ratio'],
       distribution=dist)
def set_etoac_solvent_to_crude_ratio(i):
    etoac_purification['solvent_to_crude_ratio'] = i


# Ethyl acetate price
dist = shape.Uniform(lower = F.EthylAcetate_in.price * (1-var_50), upper = F.EthylAcetate_in.price * (1+var_50))
@param(name = 'Ethyl acetate price',
    element = 'OP', 
    kind = 'isolated',
    units = 'USD/kg',
    baseline = F.EthylAcetate_in.price, distribution = dist)
def set_ethyl_acetate_price(i):
    F.EthylAcetate_in.price = i  

# Hexane solvent to pure RCF oil ratio
dist = shape.Uniform(lower=1, upper = 5)
@param(name='Hexane solvent to pure oil ratio',
       element='MP',
       kind='coupled',
       units='kg/kg',
       baseline=hexane_purification['solvent_to_oil_ratio'],
       distribution=dist)
def set_hexane_to_pure_rcf_oil_ratio(i):
    hexane_purification['solvent_to_oil_ratio'] = i

# Hexane price
dist = shape.Triangle(lower = F.Hexane_In.price * (1-var_50), midpoint = F.Hexane_In.price, upper = F.Hexane_In.price * (1+var_50))
@param(name = 'Hexane price',
    element = 'MP', 
    kind = 'isolated',
    units = 'USD/kg',
    baseline = F.Hexane_In.price, distribution = dist)
def set_hexane_price(i):
    F.Hexane_In.price = i      



In [14]:
metric = model.metric
@metric(name='Minimum Jet Selling Price', element='TEA', units='USD/gal')
def get_msp():
    msp = (integrated_tea.solve_price(F.MON_MONOMERS_OUT))
    return msp


In [15]:
import numpy as np
np.random.seed(6199)
samples = model.sample(N=3000, rule = 'L')  # Change this to 3000 later
model.load_samples(samples)

In [16]:
model.evaluate()

c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\_pump.py:224: RuntimeWarning: <Pump: RCF_PUMP1> no pump type available at current power (2.3e+03 hp), head (3.38e+03 ft), kinematic viscosity (6.07e-07 m2/s), and NPSH (1.31 ft); assuming centrigugal pump
  warn(f'{repr(self)} no pump type available at current power '
c:\users\hwadg\onedrive - the pennsylvania state university\shi_wadgama_shared\models\atjspk\lignin_saf\ligsaf_units.py:410: CostWarning: <SolvolysisReactor: RCF_RXR1> Vertical vessel length (64.26 ft) is out of bounds (12 to 40 ft) for cost correlation
  self._vertical_vessel_design(
c:\users\hwadg\onedrive - the pennsylvania state university\shi_wadgama_shared\models\atjspk\lignin_saf\ligsaf_units.py:659: CostWarning: <HydrogenolysisReactor: RCF_RXR2> Vertical vessel length (41.55 ft) is out of bounds (12 to 40 ft) for cost correlation
  self._vertical_vessel_design(
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\_unit.py:1241: CostWa

In [17]:
df_rho, df_p = model.spearman_r()
print(df_rho["TEA", "Minimum jet selling price [USD/gal]"])

Element             Parameter                               
Overall             Operating days [days]                       -0.0389
                    Poplar feedstock price [USD/DMT]             0.0638
                    Labor cost [USD/yr]                          0.0461
                    Electricity cost [USD/kWh]                  0.00476
                    Hydrogen price [USD/kg]                      0.0371
                    Hydrogen storage period [days]               0.0559
RCF                 Solvolysis reaction time [hr]                 0.296
                    Solvolysis residence time [hr]               -0.379
                    Solvolysis cleaning time [hr]              -0.00172
                    Methanol cost [USD/kg]                       0.0987
                    RCF solvent losses [-]                       0.0448
                    RCF reactor duty [-]                         0.0209
                    RCF catalyst price [USD/kg]                 -0.0126
   

In [19]:
rho = df_rho["TEA", "Minimum jet selling price [USD/gal]"]


In [21]:
msp_values = model.table["TEA", "Minimum jet selling price [USD/gal]"]

In [24]:
msp_values.quantile(0.95)

31.590819009650712

In [ ]:
model.table

Element               Overall                                                                                                          ...  \
Feature Operating days [days] Poplar feedstock price [USD/DMT] Labor cost [USD/yr] Electricity cost [USD/kWh] Hydrogen price [USD/kg]  ...   
0                         332                             82.6            6.48e+06                     0.0713                    4.81  ...   
1                         352                             61.9            8.79e+06                      0.103                    9.51  ...   
2                         339                             86.6            1.65e+07                     0.0732                    8.56  ...   
3                         334                             82.1            1.51e+07                      0.128                    5.72  ...   
4                         326                             76.4            1.84e+07                     0.0863                    7.87  ...   
...                       ...                              ...                 ...                        ...                     ...  ...   
2995                      325                             93.6            1.46e+07                      0.104                    3.53  ...   
2996                      329                             54.7            1.72e+07                       0.14                    7.19  ...   
2997                      316                             66.2            1.02e+07                     0.0867                    5.36  ...   
2998                      338                               88            6.43e+06                       0.15                    9.18  ...   
2999                      333                             87.6             1.2e+07                     0.0805                    3.39  ...   

Element                                   OP                                                                    MP                        \
Feature Et oac solvent to crude ratio [L/kg] Ethyl acetate price [USD/kg] Hexane solvent to pure oil ratio [kg/kg] Hexane price [USD/kg]   
0                                      1.12                          0.47                                2.92                      0.782   
1                                     0.943                          1.09                                4.19                       1.55   
2                                     0.999                         0.752                                   2                       1.23   
3                                      1.27                          1.07                                4.84                       1.06   
4                                      1.05                          1.11                                4.35                       1.22   
...                                     ...                           ...                                 ...                        ...   
2995                                   1.26                         0.566                                2.85                       1.09   
2996                                   1.11                          0.57                                4.66                       1.38   
2997                                   0.92                          1.13                                3.93                       1.15   
2998                                  0.958                         0.675                                2.88                      0.944   
2999                                   1.21                         0.983                                1.46                      0.981   

Element                                 TEA  
Feature Minimum jet selling price [USD/gal]  
0                                      6.25  
1                                      9.28  
2                                       4.5  
3                                      9.79  
4                                      14.7  
...       